# AnimTOON 3B Training on SageMaker (plain HF stack, no Unsloth)

## Prereqs
1. Instance: `sc.g6.xlarge` (L4, 24 GB VRAM, ~$1.13/hr)
2. Image: **PyTorch 2.x CUDA** kernel (pick the newest PyTorch GPU image when launching)
3. HF token: **hardcoded in Cell 3** (edit if rotating)
4. Training data: pulled from `srk0102200/animtoon-iconscout-v4` on HF
5. Base model: pulled from `srk0102200/AnimTOON-3B-v4` on HF (private, uses hardcoded token)

## Cost target
- Setup + deps install: ~10 min
- Training 9,992 steps (2 epochs): ~3-4 hours on L4
- **Total ~4 hrs × $1.127/hr = ~$4.50**

## Cell 1 — Install dependencies

In [ ]:
%%capture
!pip install -q -U transformers==4.46.3 peft==0.13.2 trl==0.12.0 accelerate==1.1.1 bitsandbytes==0.44.1 datasets huggingface_hub sentencepiece

## Cell 2 — Pick base model + hyperparameters

In [ ]:
# === CHANGE THIS to switch between experiments ===
BASE_MODEL = "v4"  # "v3" or "v4"

MODEL_MAP = {
    "v3": "srk0102200/AnimTOON-3B",
    "v4": "srk0102200/AnimTOON-3B-v4",
}
MODEL_NAME = MODEL_MAP[BASE_MODEL]
OUTPUT_NAME = f"animtoon-3b-{BASE_MODEL}1-adapter"
OUTPUT_DIR = f"/home/sagemaker-user/{OUTPUT_NAME}"

MAX_SEQ_LEN = 2048
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
EPOCHS = 2
BATCH_SIZE = 4          # L4 has 24GB, can fit more than T4
GRAD_ACCUM = 4          # effective batch = 16
LEARNING_RATE = 2e-4
WARMUP_RATIO = 0.05

print(f"Base model : {MODEL_NAME}")
print(f"Output dir : {OUTPUT_DIR}")

## Cell 3 — HF login + GPU check

In [ ]:
import os
import torch

# HF token for private repos. Options:
#   1. Export HF_TOKEN env var in the shell BEFORE launching Jupyter
#   2. Paste into HF_TOKEN_OVERRIDE below in the cloud editor (don't commit)
HF_TOKEN_OVERRIDE = ""  # <-- paste token here in the cloud editor, leave blank in git

if HF_TOKEN_OVERRIDE:
    os.environ["HF_TOKEN"] = HF_TOKEN_OVERRIDE
assert os.environ.get("HF_TOKEN"), (
    "HF_TOKEN not set. Paste into HF_TOKEN_OVERRIDE or `export HF_TOKEN=hf_...` in shell."
)

from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])
print("Logged into HuggingFace")

# GPU sanity check
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    print(f"Compute capability: {torch.cuda.get_device_capability(0)}")
    print(f"bf16 supported: {torch.cuda.is_bf16_supported()}")

## Cell 4 — Load base model (4-bit QLoRA)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Use bf16 compute if supported (L4 is Ada Lovelace — yes), else fp16
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading {MODEL_NAME} in 4-bit...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    token=os.environ["HF_TOKEN"],
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    padding_side="left",
    token=os.environ["HF_TOKEN"],
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model.config.use_cache = False
model.config.pretraining_tp = 1
print(f"Model loaded. Tokenizer vocab: {len(tokenizer)}")

## Cell 5 — Attach LoRA adapter

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Cell 6 — Load training data from HF

In [ ]:
import json
from pathlib import Path
from datasets import Dataset
from huggingface_hub import hf_hub_download

DATA_REPO = "srk0102200/animtoon-iconscout-v4"
DATA_FILE = "iconscout_training_v4.jsonl"

print(f"Downloading {DATA_FILE} from {DATA_REPO} ...")
data_path = hf_hub_download(
    repo_id=DATA_REPO,
    filename=DATA_FILE,
    repo_type="dataset",
    token=os.environ["HF_TOKEN"],
)
print(f"Loaded from: {data_path}")

records = []
with open(data_path, encoding="utf-8") as f:
    for line in f:
        r = json.loads(line)
        records.append({"prompt": r["prompt"], "output": r["output"]})
print(f"Loaded {len(records)} training records")
print(f"Sample prompt: {records[0]['prompt'][:100]}")
print(f"Sample output (first 200 chars):\n{records[0]['output'][:200]}")

In [ ]:
# Format using chat template + tokenize
def format_record(r):
    messages = [
        {"role": "user",      "content": f"Generate AnimTOON animation: {r['prompt']}"},
        {"role": "assistant", "content": r["output"]},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    return {"text": text}

dataset = Dataset.from_list(records)
dataset = dataset.map(format_record, remove_columns=dataset.column_names)

# 95/5 split
split = dataset.train_test_split(test_size=0.05, seed=42)
train_dataset = split["train"]
eval_dataset = split["test"]

print(f"Train: {len(train_dataset)}")
print(f"Eval:  {len(eval_dataset)}")

sample = train_dataset.select(range(min(500, len(train_dataset))))
lengths = [len(tokenizer(x["text"])["input_ids"]) for x in sample]
print(f"Token length on 500 samples — min={min(lengths)} max={max(lengths)} avg={sum(lengths)/len(lengths):.0f}")

## Cell 7 — Set up SFTTrainer

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    packing=False,
    args=SFTConfig(
        output_dir=OUTPUT_DIR,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=EPOCHS,
        learning_rate=LEARNING_RATE,
        warmup_ratio=WARMUP_RATIO,
        lr_scheduler_type="cosine",
        weight_decay=0.01,
        max_grad_norm=1.0,
        optim="paged_adamw_8bit",
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        logging_steps=25,
        save_strategy="steps",
        save_steps=500,
        save_total_limit=2,
        eval_strategy="steps",
        eval_steps=500,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        report_to="none",
        dataset_text_field="text",
        seed=42,
    ),
)
print("Trainer ready. Starting training...")

## Cell 8 — Train (~3-4 hours)

In [ ]:
trainer_stats = trainer.train()
print(f"\n✓ Training complete. Final loss: {trainer_stats.training_loss:.4f}")

## Cell 9 — Save LoRA adapter + tokenizer

In [ ]:
final_path = f"{OUTPUT_DIR}/final"
model.save_pretrained(final_path)
tokenizer.save_pretrained(final_path)
print(f"Saved LoRA adapter + tokenizer to: {final_path}")
!ls -lh {final_path}

## Cell 10 — Inference sanity check

In [ ]:
model.eval()

test_prompts = [
    "a red circle pulsing in the center",
    "a businessman waving hello with his right arm",
    "14-layer cat with head, body, 4 legs, tail. tail wags, ears twitch",
    "a person walking forward with swinging arms and stepping legs",
]

for p in test_prompts:
    messages = [{"role": "user", "content": f"Generate AnimTOON animation: {p}"}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=1024, temperature=0.5,
            top_p=0.9, do_sample=True, pad_token_id=tokenizer.eos_token_id,
        )
    result = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"\n=== {p} ===")
    print(result[:500])

## Cell 11 — Zip adapter for download

Right-click the resulting zip in the file browser → Download to your local machine.

In [ ]:
import shutil
zip_path = shutil.make_archive(
    base_name=f"/home/sagemaker-user/{OUTPUT_NAME}",
    format="zip",
    root_dir=f"{OUTPUT_DIR}/final",
)
size_mb = os.path.getsize(zip_path) / 1024 / 1024
print(f"\n✓ Adapter zipped: {zip_path} ({size_mb:.1f} MB)")
print(f"\nDownload this file to local: {zip_path}")